# R3-U2: AST inference adapter and bundle
Select a Colab **T4 GPU** runtime. This packages the checkpoint named by the frozen selection of the finished R3-U2 run (`selection.json`) into a verified, uncalibrated bundle, and checks the bundle adapter against the exported selection-role probabilities from raw audio. It trains nothing and changes no selection. Sync the new code first (see the list printed by the check cell if anything is missing).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
RUN_ID = 'r3_u2_20261112T134049Z'  # The finished R3-U2 run whose frozen selection is packaged.
RUN_DIR = f'runs/r3/{RUN_ID}'
SHARDS = ROOT / 'data/processed/cohort_v1'
OWNER = 'Subhan'
MODEL_CACHE = '/content/hf_cache'  # The pinned AST is downloaded here and hash-verified; Colab-local.
BUNDLE = 'bundles/r3_u2_ast_partial_B'  # A new folder: the script refuses to replace an existing bundle.
assert (ROOT / 'scripts/r3_u2_ast_bundle.py').is_file(), 'Sync the updated project code to Drive'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, str(ROOT / 'src'))  # lets this kernel import echoproof, not only the subprocesses

In [ ]:
import torch, transformers, numpy
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| numpy', numpy.__version__)
assert torch.cuda.is_available(), 'Choose a GPU runtime (the parity gate runs the adapter in fp16 on CUDA)'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()
run = ROOT / RUN_DIR
selection = json.loads((run / 'selection.json').read_text(encoding='utf-8'))
winner = selection['selected']
checkpoint = run / f'{RUN_ID}-{winner}' / 'best.pt'
required = ['configs/r3_u2_ast_ft_b.json', 'src/echoproof/models/ast_adapter.py', 'src/echoproof/models/r3_u2_bundle.py',
            'src/echoproof/training/r3_u2_parity.py', 'src/echoproof/training/r3_u2_data.py', 'tests/test_r3_u2_adapter.py',
            'data/manifests/cohort_v1.parquet', 'data/splits/protocol_B_v1.parquet', 'data/processed/chunks_v1.parquet',
            'data/features/r2_s1_B/r2_s1_manifest.json', 'reports/r1/split_report.json', RUN_DIR + '/predictions_chunks_selection.parquet']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing inputs: {missing}'
assert checkpoint.is_file() and sha256(checkpoint) == selection['checkpoint_sha256'][winner], 'The selected checkpoint is missing or changed'
assert SHARDS.is_dir() and not (ROOT / BUNDLE).exists(), 'Missing shards, or the bundle folder already exists'
print('Selected:', winner, '| epoch', selection['selected_epoch'], '| checkpoint', selection['checkpoint_sha256'][winner][:12])

## Fixture tests
Any failure or skip must be resolved first. They cover packaging refusals, the round trip through the bundle, the adapter contract and the parity helpers on a tiny random AST.

In [ ]:
tests = ['tests/test_r3_u2_adapter.py', 'tests/test_r3_u2_ast.py', 'tests/test_r1_u2_ast.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *tests], text=True, capture_output=True)
print(check.stdout[-4000:], check.stderr[-2000:])
assert check.returncode == 0 and ' skipped' not in check.stdout, 'Resolve test failures or skips first'

## Package
Loads the selected weights strictly, checks that the frozen groups equal the pinned pretrained AST and the unfrozen groups differ, re-saves as safetensors with a two-class config and the pinned extractor config, and verifies the conversion tensor by tensor. The pinned AST is downloaded and hash-verified.

In [ ]:
subprocess.run([sys.executable, 'scripts/r3_u2_ast_bundle.py', 'package', '--root', str(ROOT), '--run-dir', RUN_DIR, '--owner', OWNER,
                '--bundle', BUNDLE, '--model-cache', MODEL_CACHE], check=True)

## Adapter parity
Scores 50 deterministic selection-role chunks from the raw shard audio. Gate: the bundle adapter in fp16 on CUDA reproduces the exported probabilities within 0.01. Reported, not gated: the adapter in fp32 on CUDA and on CPU, which shows how far CPU inference sits from the scores R3-N1 will calibrate.

In [ ]:
subprocess.run([sys.executable, 'scripts/r3_u2_ast_bundle.py', 'parity', '--root', str(ROOT), '--run-dir', RUN_DIR, '--owner', OWNER,
                '--bundle', BUNDLE, '--shard-dir', str(SHARDS)], check=True)

## Verify and write the report

In [ ]:
import numpy as np
from echoproof.bundle import load_bundle
from echoproof.models.ast_adapter import ASTBundleAdapter
bundle = load_bundle(ROOT / BUNDLE)
parity_path = sorted((ROOT / 'reports/r3').glob('r3_u2_ast_parity_*.json'))[-1]
parity = json.loads(parity_path.read_text(encoding='utf-8'))
assert parity['passed'] is True and parity['experiment'] == winner and parity['checkpoint_sha256'] == selection['checkpoint_sha256'][winner]
adapter = ASTBundleAdapter.load(ROOT / BUNDLE, device='cuda', precision='fp16')
probe = adapter.score_chunks([np.random.default_rng(0).normal(0, 0.05, 16000).astype(np.float32)])
assert probe.shape == (1,) and 0 <= probe[0] <= 1
print('bundle:', bundle.spec.bundle_id, '| model_version:', bundle.spec.model_version)
print('fp16 CUDA vs exports:', parity['fp16'])
for name, value in parity['reported_not_gated'].items():
    print(name, value)
report = {'task': 'R3-U2', 'kind': 'ast_bundle', 'run_id': RUN_ID, 'bundle': str(ROOT / BUNDLE), 'selected': winner,
          'bundle_json_sha256': sha256(ROOT / BUNDLE / 'bundle.json'), 'parity_report': parity_path.name, 'parity_report_sha256': sha256(parity_path),
          'parity': {k: parity[k] for k in ('items', 'passed', 'atol_fp16', 'fp16', 'reported_not_gated')},
          'gpu': torch.cuda.get_device_name(0), 'verification': 'tests, bundle verification, conversion check and parity passed; independent review pending'}
report_path = ROOT / 'reports/r3' / ('r3_u2_ast_bundle_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '.json')
with report_path.open('x', encoding='utf-8') as f:
    json.dump(report, f, indent=2, sort_keys=True)
print('Report written:', report_path)